# Category1 vs Category2 Routing Comparison

Start small: load the 102-string inference prediction CSVs, load LIW weights, build the same `final_weight` convention used in `Metadata/EffectiveArea/plots.ipynb`, and merge weights into both prediction tables.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path("/project/def-nahee/kbas/Graphnet-Applications")
RESULTS_ROOT = REPO_ROOT / "Results/340StringMC/102_string_emax1e6"
WEIGHT_DIR = REPO_ROOT / "Metadata/EventWeights/String340MC"

CAT1_CSV = RESULTS_ROOT / "inference/first_category/exp001/inference/inference_predictions.csv"
CAT2_CSV = RESULTS_ROOT / "inference/second_category/exp002/inference/inference_predictions.csv"

WEIGHT_FILES = {
    "electron": WEIGHT_DIR / "Electron_LIW.csv",
    "muon": WEIGHT_DIR / "Muon_LIW.csv",
    "nc": WEIGHT_DIR / "NC_LIW.csv",
    "tau": WEIGHT_DIR / "Tau_LIW.csv",
}

JOIN_COLUMNS = ["RunID", "SubrunID", "EventID", "SubEventID"]
ENERGY_BINS_LOG10 = np.array([2, 3, 4, 5, 6], dtype=float)
ENERGY_BIN_LABELS = ["2-3", "3-4", "4-5", "5-6"]

## Load 102-String Predictions

In [ ]:
cat1 = pd.read_csv(CAT1_CSV)
cat2 = pd.read_csv(CAT2_CSV)

print(f"Category1 rows: {len(cat1):,} | columns: {len(cat1.columns)}")
print(f"Category2 rows: {len(cat2):,} | columns: {len(cat2.columns)}")

cat1.head()

In [ ]:
cat2.head()

## Load LIW Weights

`Metadata/EffectiveArea/plots.ipynb` defines analysis weights by splitting each flavor into neutrino and antineutrino samples, then using:

`oneweight = oneweight_x100 / len(flavor_particle_sample)`

`final_weight = oneweight * survivalProb`

In [ ]:
def add_event_type(df):
    out = df.copy()
    pid_abs = out["pid"].abs() if "pid" in out else out["initialType"].abs()
    interaction = out["interaction_type"] if "interaction_type" in out else np.where(out["finalType1"].abs().isin([12, 14, 16]), 2, 1)

    out["true_event_type"] = "other"
    out.loc[(pid_abs == 14) & (interaction == 1), "true_event_type"] = "muon_CC"
    out.loc[(pid_abs == 12) & (interaction == 1), "true_event_type"] = "electron_CC"
    out.loc[(pid_abs == 16) & (interaction == 1), "true_event_type"] = "tau_CC"
    out.loc[interaction == 2, "true_event_type"] = "NC"
    return out


def split_particle_type(weights):
    return {
        "neutrino": weights.loc[weights["initialType"] > 0].copy(),
        "antineutrino": weights.loc[weights["initialType"] < 0].copy(),
    }


def add_analysis_weights(sample, flavor, particle_type):
    weighted = sample.copy()
    weighted["flavor"] = flavor
    weighted["particle_type"] = particle_type
    weighted["oneweight_analysis"] = weighted["oneweight_x100"] / len(weighted)
    weighted["final_weight"] = weighted["oneweight_analysis"] * weighted["survivalProb"]
    return weighted


weight_parts = []
for flavor, path in WEIGHT_FILES.items():
    weights = pd.read_csv(path)
    for particle_type, sample in split_particle_type(weights).items():
        weight_parts.append(add_analysis_weights(sample, flavor, particle_type))

weights = pd.concat(weight_parts, ignore_index=True)
weights = weights.drop_duplicates(subset=JOIN_COLUMNS, keep="first")

weights[[*JOIN_COLUMNS, "flavor", "particle_type", "oneweight_analysis", "survivalProb", "final_weight"]].head()

## Merge Weights Into Predictions

In [ ]:
WEIGHT_COLUMNS = [
    *JOIN_COLUMNS,
    "flavor",
    "particle_type",
    "oneweight",
    "oneweight_x100",
    "oneweight_analysis",
    "survivalProb",
    "final_weight",
]


def merge_weights(predictions, label):
    merged = predictions.merge(weights[WEIGHT_COLUMNS], on=JOIN_COLUMNS, how="left", indicator="weight_merge")
    merged = add_event_type(merged)
    matched = merged["final_weight"].notna().sum()
    missing = merged["final_weight"].isna().sum()
    print(f"{label}: rows={len(merged):,} matched_weights={matched:,} missing_weights={missing:,}")
    return merged


cat1_w = merge_weights(cat1, "Category1")
cat2_w = merge_weights(cat2, "Category2")

In [ ]:
def effective_sample_size(weight):
    w = pd.Series(weight).dropna().to_numpy(dtype=float)
    w = w[w > 0]
    if len(w) == 0:
        return np.nan
    return (w.sum() ** 2) / np.sum(w ** 2)


def merge_quality_table(datasets):
    rows = []
    for label, df in datasets.items():
        has_weight = df["final_weight"].notna()
        true_muons = df["true_event_type"].eq("muon_CC")
        rows.append({
            "dataset": label,
            "rows": len(df),
            "matched_weight_rows": int(has_weight.sum()),
            "missing_weight_rows": int((~has_weight).sum()),
            "true_muons_with_weights": int((has_weight & true_muons).sum()),
            "sum_final_weight": df.loc[has_weight, "final_weight"].sum(),
            "n_eff": effective_sample_size(df.loc[has_weight, "final_weight"]),
        })
    return pd.DataFrame(rows)


merge_quality = merge_quality_table({"category1": cat1_w, "category2": cat2_w})
merge_quality

## First Sanity Checks

In [ ]:
def event_type_summary(df, weight_column="final_weight"):
    unweighted = df.groupby("true_event_type", dropna=False).size().rename("n_events")
    weighted = df.groupby("true_event_type", dropna=False)[weight_column].sum(min_count=1).rename("sum_final_weight")
    return pd.concat([unweighted, weighted], axis=1).reset_index()


event_type_summary(cat1_w)

In [ ]:
event_type_summary(cat2_w)